## Importing Packages

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns',150)
pd.set_option('display.max_rows',150)

## Importing Decision Engine Output

In [3]:
df = pd.read_pickle("../data/decision_engine_output.pkl")
df.shape

(40000, 153)

In [4]:
df[["loan_decision","pd_score","lgd_estimate","ead","expected_loss","expected_interest_income",
    "risk_adjusted_return", "risk_band", "actual_default_flag"]].head()

,loan_decision,pd_score,lgd_estimate,ead,expected_loss,expected_interest_income,risk_adjusted_return,risk_band,actual_default_flag
0,Reject,0.495415,0.695648,12000,4135.613883,2386.8000,-1748.813883,High,1
1,Reject,0.504460,0.611491,20100,6200.299219,3133.5900,-3066.709219,High,0
2,Reject,0.301881,0.558952,8000,1349.895463,891.2000,-458.695463,Low,0
3,Reject,0.685799,0.611491,23325,9781.568456,2959.9425,-6821.625956,Very High,0
4,Reject,0.739917,0.716095,21250,11259.330155,4876.8750,-6382.455155,Very High,0


## Eligible Loans

In [10]:
eligible_loans = df[df["loan_decision"].isin(["Approve","Manual Review"])].copy()
eligible_loans["loan_decision"].value_counts()

loan_decision
Manual Review    8000
Approve          3799
Name: count, dtype: int64

In [11]:
eligible_loans = eligible_loans.sort_values(by='risk_adjusted_return', ascending=False)
eligible_loans['portfolio_rank'] = range(1, len(eligible_loans) + 1)

In [12]:
eligible_loans[['portfolio_rank',"loan_decision","pd_score","expected_loss","expected_interest_income",
    "risk_adjusted_return"]].head()

,portfolio_rank,loan_decision,pd_score,expected_loss,expected_interest_income,risk_adjusted_return
18722,1,Approve,0.096850,1894.716335,4242.0,2347.283665
38246,2,Approve,0.126078,2466.501270,4546.5,2079.998730
39601,3,Approve,0.100250,1737.079032,3757.2,2020.120968
21374,4,Approve,0.162944,3187.733785,4931.5,1743.766215
11429,5,Approve,0.052854,982.389058,2621.5,1639.110942


## Selecting Optimized Portfolio

In [13]:
size=5000
optimized_portfolio = eligible_loans.head(size).copy()
print(f'Optimized Portfolio Shape {optimized_portfolio.shape}')

Optimized Portfolio Shape (5000, 154)


## Portfolio_summary

In [17]:
def portfolio_summary(data, portfolio_name):
    return {
        'portfolio':portfolio_name,
        'borrower_count': len(data),
        'actual_default_rate': data['actual_default_flag'].mean(),
        'avg_pd': data['pd_score'].mean(),
        'avg_lgd': data['lgd_estimate'].mean(),
        'total_ead': data['ead'].sum(),
        'avg_ead' : data['ead'].mean(),
        'total_expected_loss': data['expected_loss'].sum(),
        'avg_expected_loss' : data['expected_loss'].mean(),
        'total_expected_interest_income' : data['expected_interest_income'].sum(),
        'avg_expected_interest_income' : data['expected_interest_income'].mean(),
        'total_risk_adjusted_return' : data['risk_adjusted_return'].sum(),
        'avg_risk_adjusted_return' : data['risk_adjusted_return'].mean()
             }

In [20]:
portfolio_comparision = pd.DataFrame([portfolio_summary(eligible_loans, 'Eligible Portfolio'), 
                                     portfolio_summary(optimized_portfolio, 'Optimized Portfolio')])
portfolio_comparision

,portfolio,borrower_count,actual_default_rate,avg_pd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_expected_interest_income,avg_expected_interest_income,total_risk_adjusted_return,avg_risk_adjusted_return
0,Eligible Portfolio,11799,0.137469,0.326267,0.581394,160863325,13633.640563,3.105600e+07,2632.087337,1.896885e+07,1607.665941,-1.208715e+07,-1024.421396
1,Optimized Portfolio,5000,0.076800,0.192400,0.561357,56814975,11362.995000,4.634156e+06,926.831193,5.074427e+06,1014.885305,4.402706e+05,88.054113
